<a href="https://colab.research.google.com/github/AliTabatabaiGWU/HW2_Test_SEAS8515/blob/main/HW2_FashionMNIST_Tabatabaib.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# load and process data

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import fashion_mnist
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns

# 1. Load the dataset
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data() # [cite: 112]

# 2. Normalize pixel values to be between 0.0 and 1.0
X_train = X_train / 255.0
X_test = X_test / 255.0

# 3. Define the class names for plotting later
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'] #[cite: 108]

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")


# Architecture 1 shallow baseline

print("--- Training Model 1 ---")

# Build Model 1
model_1 = models.Sequential([
    layers.Flatten(input_shape=(28, 28)), # Flattens 2D images to 784 vectors
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax') # 10 classes outputting a probability distribution
])

# Compile using sparse categorical crossentropy so we don't have to one-hot encode labels
model_1.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

# Print total parameter count
model_1.summary() # [cite: 117]

# Train the model
history_1 = model_1.fit(X_train, y_train,
                        epochs=10,
                        validation_split=0.2, # Holds out 20% of training data for validation #[cite: 115]
                        batch_size=64)

# Evaluate on the test set
test_loss_1, test_acc_1 = model_1.evaluate(X_test, y_test, verbose=0) #[cite: 119]
print(f"\nModel 1 Final Test Accuracy: {test_acc_1:.4f}") #[cite: 119]

# Architecture 2 - Deeper with Regularization

print("--- Training Model 2 ---")

# Build Model 2
model_2 = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(256, activation='relu'),   # Increased width
    layers.Dropout(0.3),                    # Regularization step to prevent memorizing noise
    layers.Dense(128, activation='relu'),   # Increased depth
    layers.Dense(10, activation='softmax')
])

model_2.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

# Print total parameter count
model_2.summary() #[cite: 117]

# Train the model
history_2 = model_2.fit(X_train, y_train,
                        epochs=10,
                        validation_split=0.2,
                        batch_size=64)

# Evaluate on the test set
test_loss_2, test_acc_2 = model_2.evaluate(X_test, y_test, verbose=0) [cite: 119]
print(f"\nModel 2 Final Test Accuracy: {test_acc_2:.4f}") #[cite: 119]

# Plot learning curves

# Change this to history_1 if Model 1 performed better
best_history = history_2

epochs_range = range(1, 11)

plt.figure(figsize=(14, 5))

# Plot 1: Accuracy Curves
plt.subplot(1, 2, 1)
plt.plot(epochs_range, best_history.history['accuracy'], label='Training Accuracy')
plt.plot(epochs_range, best_history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training vs. Validation Accuracy')
plt.legend()

# Plot 2: Loss Curves
plt.subplot(1, 2, 2)
plt.plot(epochs_range, best_history.history['loss'], label='Training Loss')
plt.plot(epochs_range, best_history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training vs. Validation Loss')
plt.legend()

# Save the plot automatically to your Colab directory
plt.savefig('best_model_learning_curves.png') #[cite: 121]
plt.show()

# confusion matrix heatmap

# Change this to model_1 if Model 1 was your best performer
best_model = model_2

# 1. Generate predictions (probabilities)
y_pred_probs = best_model.predict(X_test)

# 2. Get the index of the highest probability for each sample
y_pred = np.argmax(y_pred_probs, axis=1)

# 3. Create the confusion matrix
cm = confusion_matrix(y_test, y_pred) #[cite: 122]

# 4. Plot using Seaborn
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('Actual Class')
plt.xlabel('Predicted Class')
plt.title('Confusion Matrix - Best Model')
plt.show()

Training data shape: (60000, 28, 28)
Test data shape: (10000, 28, 28)
--- Training Model 1 ---


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_1 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_2 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 235,146 (918.54 KB)

 Trainable params: 235,146 (918.54 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.8000 - loss: 0.5570 - val_accuracy: 0.8553 - val_loss: 0.4101
Epoch 2/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8494 - loss: 0.4148 - val_accuracy: 0.8658 - val_loss: 0.3633
Epoch 3/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.8608 - loss: 0.3816 - val_accuracy: 0.8684 - val_loss: 0.3611
Epoch 4/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8662 - loss: 0.3587 - val_accuracy: 0.8766 - val_loss: 0.3397
Epoch 5/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.8740 - loss: 0.3420 - val_accuracy: 0.8789 - val_loss: 0.3339
Epoch 6/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.8763 - loss: 0.3320 - val_accuracy: 0.8832 - val_loss: 0.3268
Epoch 7/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8819 - loss: 0.3167 - val_accuracy: 0.8785 - val_loss: 0.3366
Epoch 8/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8844 - loss: 0.3095 - val_accuracy: 0.